In [ ]:
import nibabel as nib
import ants
import numpy as np
import matplotlib.pyplot as plt
import os
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, fixed
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from mpl_toolkits.axes_grid1 import ImageGrid

In [ ]:
# Para garantir que os plots apareçam no notebook
%matplotlib inline 

def plot_slice(image_3d, slice_index, label='', dataset='', axis=2, rot=1):
    # Seleciona e prepara a fatia 2D
    if axis == 0:
        image_slice_2d = image_3d[slice_index, :, :]
    elif axis == 1:
        image_slice_2d = image_3d[:, slice_index, :]
    else:
        image_slice_2d = image_3d[:, :, slice_index]
    image_slice_2d = np.rot90(image_slice_2d, k=rot)
    
    # Cria a plotagem
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(image_slice_2d, cmap='gray')
    
    # Título principal
    title_text = f"Dataset {dataset} | Label: {label}"
    ax.set_title(title_text, fontsize=16)
    
    # Título da fatia
    ax.text(5, 15, f"Fatia: {slice_index}/{image_3d.shape[2]}", color='white', 
            bbox=dict(facecolor='black', alpha=0.5))
    
    ax.axis('off')
    plt.show()

    # metricas_imagem(image_3d)

def view_interactive_slices(image_3d, y_true, dataset, axis=2, rot=1):
    num_slices = image_3d.shape[2]

    # Criar e conectar o slider à função de plotagem
    interact(plot_slice, image_3d=fixed(image_3d), label=fixed(y_true), dataset=fixed(dataset), axis=fixed(axis), rot=fixed(rot),
             slice_index=IntSlider(value=num_slices//2, min=0, max=num_slices-1, step=1, description='Fatia:'))
    
def load_nifti_data_balanced(base_dir, class_names, target=1000):
    images = []
    labels = []
    paths = []
    
    # Caminhos das subpastas
    for label in class_names:
        print(f"carregando diretório {label}")
        label_dir = os.path.join(base_dir, label)
        count = 0

        names = os.listdir(label_dir)
        #np.random.shuffle(names)
        for fname in names:
            if count < target:
                img_path = os.path.join(label_dir, fname)
                img = nib.load(img_path).get_fdata(dtype=np.float16)
                paths.append(img_path)
                # print(img.shape)
                images.append(img)
                labels.append(label)
                count += 1

        print(f"diretório carregado {count}")

    # Codificando os rótulos
    label_encoder = LabelEncoder()
    label_encoder.classes_ = np.array(class_names)
    labels_encoded = label_encoder.transform(labels)
    labels_one_hot = to_categorical(labels_encoded, num_classes=len(class_names))

    # Convertendo para arrays NumPy
    images = np.array(images).reshape((-1, *images[0].shape, 1))
    labels_one_hot = np.array(labels_one_hot)
    
    # Embaralhar os dados
    # images, labels_one_hot, paths = shuffle(images, labels_one_hot, paths, random_state=42)
    
    return images, labels_one_hot, paths, label_encoder.classes_

def metricas_imagem(data, title): #print metricas de uma imagem (max, min, media)
    if len(data.shape) > 1:
        values = data.flatten()
    else:
        values = data

    # print(f"MEDIA: {np.mean(values)}")
    # print(f"DESVIO: {np.std(values)}")
    # print(f"MIN: {np.min(values)}")
    # print(f"MAX: {np.max(values)}")

    plt.hist(values)
    plt.title(f"{title} | Média: {np.mean(values)}")
    plt.show()

def unificar_tamanhos_com_padding(lista_de_imagens):
    max_altura = 0
    max_largura = 0
    for img in lista_de_imagens:
        altura, largura = img.shape
        if altura > max_altura:
            max_altura = altura
        if largura > max_largura:
            max_largura = largura

    imagens_uniformes = []
    for img in lista_de_imagens:
        fundo = np.zeros((max_altura, max_largura))
        
        altura_img, largura_img = img.shape
        y_offset = (max_altura - altura_img) // 2
        x_offset = (max_largura - largura_img) // 2
        
        fundo[y_offset:y_offset+altura_img, x_offset:x_offset+largura_img] = img
        imagens_uniformes.append(fundo)
        
    return imagens_uniformes

def plot_views_uniforme_final(image, main_title, k=0, sag_idx=90, cor_idx=110, ax_idx=110, figsize=(15, 5), axes_pad=0.3):
    fig = plt.figure(figsize=figsize)
    grid = ImageGrid(fig, 111,
                    nrows_ncols=(1, 3),
                    axes_pad=axes_pad)
    
    slices_originais = [
        np.rot90(image[sag_idx, :, :], k=k),
        np.rot90(image[:, cor_idx, :], k=k),
        np.rot90(image[:, :, ax_idx], k=k)
    ]
    
    slices_uniformizadas = unificar_tamanhos_com_padding(slices_originais)
    
    titles = ["Sagital", "Coronal", "Axial"]

    for ax, im_slice, title in zip(grid, slices_uniformizadas, titles):
        ax.imshow(im_slice, cmap='gray')
        ax.set_title(title)
        ax.axis('off')

    fig.suptitle(main_title)
    plt.show()

In [ ]:
# Caminhos para os diretórios e constantes
oasis_dir = "/mnt/c/Users/Bruno/Desktop/IANS/OASIS_2_PROCESSED"
adni_dir = "/mnt/c/Users/Bruno/Desktop/IANS/Alzheimer/test_affine"

raw_oasis_dir = "/mnt/c/Users/Bruno/Desktop/IANS/OASIS_2_RAW"
raw_adni_dir = "/mnt/c/Users/Bruno/Desktop/IANS/Alzheimer/test_raw"

classes_oasis = ['0.0', '0.5', '1.0']
classes_adni = ['CN', 'EMCI', 'MCI', 'LMCI', 'AD']

### DADOS PRÉ-PROCESSADOS

In [ ]:
# Carregar dados do OASIS
oasis_images, oasis_labels, oasis_paths, _ = load_nifti_data_balanced(oasis_dir, classes_oasis)

In [ ]:
# Carregar dados do ADNI
adni_images, adni_labels, adni_paths, _ = load_nifti_data_balanced(adni_dir, classes_adni)

In [ ]:
for i in range (len(adni_images[0:10])):
    view_interactive_slices(adni_images[i], classes_adni[np.argmax(adni_labels[i])], 'ADNI')
    view_interactive_slices(oasis_images[i], classes_oasis[np.argmax(oasis_labels[i])], 'OASIS')

In [ ]:
for i in range (len(adni_images[0:10])):
    metricas_imagem(adni_images[i], 'ADNI')
    metricas_imagem(oasis_images[i], 'OASIS')

### TESTANDO DADOS PRÉ-PROCESSADOS

In [ ]:
testing_adni = ants.image_read(adni_paths[0])
testing_oasis = ants.image_read(oasis_paths[0])

# view_interactive_slices(testing_adni.numpy(), classes_adni[np.argmax(adni_labels[0])], 'ADNI')
# view_interactive_slices(testing_oasis.numpy(), classes_oasis[np.argmax(oasis_labels[0])], 'OASIS')

metricas_imagem(testing_adni.numpy(), "adni")
metricas_imagem(testing_oasis.numpy(), "oasis")

In [ ]:
# Filtro de mediana
img_adni_denoised = ants.iMath(testing_adni, "MD", 1)
img_oasis_denoised = ants.iMath(testing_oasis, "MD", 1)

# view_interactive_slices(img_adni_denoised.numpy(), classes_adni[np.argmax(adni_labels[0])], 'ADNI')
# view_interactive_slices(img_oasis_denoised.numpy(), classes_oasis[np.argmax(oasis_labels[0])], 'OASIS')

metricas_imagem(img_adni_denoised.numpy(), "adni")
metricas_imagem(img_oasis_denoised.numpy(), "oasis")

In [ ]:
plot_views_uniforme_final(testing_adni.numpy(), 'ADNI original', 1, 90, 80, 82)
plot_views_uniforme_final(img_adni_denoised.numpy(), 'ADNI pós filtro 3x3', 1, 90, 80, 82)

plot_views_uniforme_final(img_oasis_denoised.numpy(), 'OASIS pós filtro 3x3', 1, 90, 80, 82)
plot_views_uniforme_final(testing_oasis.numpy(), 'OASIS original', 1, 90, 80, 82)

In [ ]:
# Testando filtro passa baixa
low_pass_adni = ants.n4_bias_field_correction(testing_adni, shrink_factor=1)
low_pass_oasis = ants.n4_bias_field_correction(testing_oasis, shrink_factor=1)

In [ ]:
plot_views_uniforme_final(testing_adni.numpy(), 'ADNI original', 1, 90, 80, 82)
plot_views_uniforme_final(low_pass_adni.numpy(), 'ADNI pós passa alta', 1, 90, 80, 82)

plot_views_uniforme_final(low_pass_oasis.numpy(), 'OASIS pós passa alta', 1, 90, 80, 82)
plot_views_uniforme_final(testing_oasis.numpy(), 'OASIS original', 1, 90, 80, 82)

In [ ]:
hist_matched_adni = ants.histogram_match_image(
     source_image=testing_adni,    
     reference_image=testing_oasis,
     number_of_histogram_bins=256
)

hist_matched_oasis = ants.histogram_match_image(
     source_image=testing_oasis,    
     reference_image=testing_adni,
     number_of_histogram_bins=256
)


In [ ]:
plot_views_uniforme_final(testing_adni.numpy(), 'ADNI original', 1, 90, 80, 82)
plot_views_uniforme_final(hist_matched_adni.numpy(), 'ADNI pós matching de histograma', 1, 90, 80, 82)

plot_views_uniforme_final(testing_oasis.numpy(), 'OASIS original', 1, 90, 80, 82)
plot_views_uniforme_final(hist_matched_oasis.numpy(), 'OASIS pós matching de histograma', 1, 90, 80, 82)

In [ ]:
# metricas_imagem(hist_matched_oasis, "oasis com match")
# metricas_imagem(testing_adni, "adni original")

# metricas_imagem(hist_matched_adni, "adni com match")
# metricas_imagem(testing_oasis, "oasis original")

### DADOS CRUS

In [ ]:
# Carregar dados não pré-processados do OASIS
raw_oasis_images, raw_oasis_labels, raw_oasis_paths, _ = load_nifti_data_balanced(raw_oasis_dir, classes_oasis)

In [ ]:
raw_classes_adni = ['CN', 'EMCI', 'LMCI', 'AD']

# Carregar dados não pré-processados do ADNI
raw_adni_images, raw_adni_labels, raw_adni_paths, _ = load_nifti_data_balanced(raw_adni_dir, raw_classes_adni, target=10)

In [ ]:
for i in range (len(raw_adni_images[0:10])):
    print("-" * 200)
    print("-" * 200)
    metricas_imagem(raw_adni_images[i], 'RAW_ADNI')
    view_interactive_slices(raw_adni_images[i], classes_adni[np.argmax(raw_adni_labels[i])], 'RAW_ADNI')
    print("-" * 200)
    print("-" * 200)
    metricas_imagem(raw_oasis_images[i], 'RAW_OASIS')
    view_interactive_slices(raw_oasis_images[i], classes_oasis[np.argmax(raw_oasis_labels[i])], 'RAW_OASIS', 1, 2)

In [ ]:
for i in range (len(raw_adni_images[0:10])):
    view_interactive_slices(raw_adni_images[i], classes_adni[np.argmax(raw_adni_labels[i])], 'RAW_ADNI')
    view_interactive_slices(raw_oasis_images[i], classes_oasis[np.argmax(raw_oasis_labels[i])], 'RAW_OASIS', 1, 2)

In [ ]:
for i in range (len(raw_adni_images[0:10])):
    metricas_imagem(raw_adni_images[i], 'RAW_ADNI')
    metricas_imagem(raw_oasis_images[i], 'RAW_OASIS')

### Testes entre os dados

In [ ]:
raw_adni = raw_adni_images[0]
raw_oasis = raw_oasis_images[0]
print(raw_adni.shape)
print(raw_oasis.shape)

plt.imshow(raw_oasis[:, :, 90])
plt.show()
plt.imshow(raw_adni[90, :, :])
plt.show()